# Session 10 — Implementing CI/CD Pipelines with GitHub Actions for MLOps

**Goal:** automate what you've been doing by hand in every prior session — running
tests, building a Docker image (Session 6), and deploying it — so it happens
automatically on every push, using GitHub Actions.

## CI vs. CD, in MLOps terms

* **CI (Continuous Integration)**: every push automatically runs tests — does the
  FastAPI app from Session 7 still pass its `TestClient` checks? Does the training
  script still run without error?
* **CD (Continuous Deployment)**: if CI passes on the main branch, automatically build
  the Docker image (Session 6) and push it to a registry, or trigger a deployment.

This session builds the *basic* pipeline — test, build, push. Session 24 goes
further with a model-quality gate that can block a deployment even when tests pass.

## Prerequisites

Needs a **GitHub repository** to actually trigger runs — the workflow YAML below is
complete and correct, and the `pytest` file it references is executed directly in
this notebook so you can verify the test logic works before wiring it into CI.

```bash
pip install pytest fastapi httpx
```

## Step 1 — Write a test file for the model API

This mirrors Session 7's FastAPI app, but as a proper `pytest` test module — the
thing CI will actually run.

In [ ]:
import os
os.makedirs("session10_ci_demo", exist_ok=True)

api_code = '''\
from fastapi import FastAPI
from pydantic import BaseModel
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import numpy as np

X, y = load_iris(return_X_y=True)
TARGET_NAMES = load_iris().target_names.tolist()
model = RandomForestClassifier(n_estimators=50, random_state=0).fit(X, y)

app = FastAPI()

class PredictRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(request: PredictRequest):
    features = np.array([[request.sepal_length, request.sepal_width,
                           request.petal_length, request.petal_width]])
    pred = model.predict(features)[0]
    return {"predicted_class": TARGET_NAMES[pred]}
'''
with open("session10_ci_demo/api.py", "w") as f:
    f.write(api_code)
print(api_code)

In [ ]:
test_code = '''\
from fastapi.testclient import TestClient
from api import app

client = TestClient(app)

def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json() == {"status": "ok"}

def test_predict_valid_input():
    response = client.post("/predict", json={
        "sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2
    })
    assert response.status_code == 200
    assert response.json()["predicted_class"] in ["setosa", "versicolor", "virginica"]

def test_predict_missing_field_returns_422():
    response = client.post("/predict", json={"sepal_length": 5.1})
    assert response.status_code == 422
'''
with open("session10_ci_demo/test_api.py", "w") as f:
    f.write(test_code)
print(test_code)

## Step 2 — Run the tests locally, exactly as CI will

If this fails locally, it will fail in CI too — running it here first is the fast
feedback loop; CI is the safety net for when someone forgets to.

In [ ]:
import subprocess

result = subprocess.run(
    ["pytest", "test_api.py", "-v"],
    cwd="session10_ci_demo",
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

## Step 3 — Write the GitHub Actions workflow

Place this at `.github/workflows/ci-cd.yml`. Three jobs: `test` runs on every push,
`build-and-push` only runs after `test` passes and only on `main`, `deploy` only runs
after a successful build — each job gates the next.

In [ ]:
workflow_yaml = '''\
name: MLOps CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements.txt
      - run: pytest test_api.py -v

  build-and-push:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: ghcr.io/${{ github.repository }}/iris-classifier:${{ github.sha }}

  deploy:
    needs: build-and-push
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - name: Deploy to Kubernetes
        run: |
          echo "kubectl set image deployment/iris-classifier \\
            iris-classifier=ghcr.io/${{ github.repository }}/iris-classifier:${{ github.sha }}"
'''
os.makedirs("session10_ci_demo/.github/workflows", exist_ok=True)
with open("session10_ci_demo/.github/workflows/ci-cd.yml", "w") as f:
    f.write(workflow_yaml)
print(workflow_yaml)

## Step 4 — Understanding what triggers what

* `on: push/pull_request` — runs `test` on every push and every PR, catching breakage
  before it merges.
* `needs: test` — `build-and-push` will not start until `test` finishes successfully;
  a failing test blocks the Docker build entirely.
* `if: github.ref == 'refs/heads/main'` — only build/deploy from `main`, never from a
  feature branch or PR, keeping untested branches from reaching production.
* `${{ github.sha }}` as the image tag — every deploy is traceable back to the exact
  commit that produced it, unlike a floating `latest` tag.

## What to try next

* Add a `poetry run` or `pip-audit` step to the `test` job to catch dependency
  vulnerabilities before they reach `main`.
* Session 24 extends this pipeline with a model-quality gate (skip deployment if a
  newly trained model's accuracy regresses versus the currently deployed one).
* Wire in the Session 3 DagsHub MLflow tracking URI as a GitHub Actions secret, so a
  training step in this workflow can log runs automatically on every push.